# Field interpolation

This short tutorial demonstrates how to use `smudgy` to interpolate a
particle field to arbitrary coordinates. We present two concise examples:

1. Interpolation of a single scalar field to a regular 2D grid of query
	 points.
2. Interpolation of the same field's gradient at the same grid points.

The examples use `smudgy.grid.create_grid_2d` to construct the 2D cell‑center
coordinates and the `PointCloud` interpolation routines. They are intentionally
compact — full, annotated notebooks belong in the tutorial directory.

In [3]:
# global setup
import numpy as np
import matplotlib.pyplot as plt
import smudgy as sm

plt.style.use("../mpl_stylefile")
%matplotlib inline
%config InlineBackend.figure_format='retina'


## Interpolating a single scalar field

This example sets up random particle data, computes smoothing information,
and interpolates a smooth test field to a regular 2D grid of query points.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import smudgy as sm
from smudgy.grid import create_grid_2d

# --- generate sample particle data ---
boxsize = 1.0
N = 1000
positions = np.random.uniform(0.0, boxsize, size=(N, 2))
weights = np.ones(N)

# a smooth test field defined on the particle positions
values = np.sin(2 * np.pi * positions[:, 0]) * np.cos(2 * np.pi * positions[:, 1])

# --- build point cloud and compute smoothing ---
pc = sm.PointCloud(positions, weights, boxsize=boxsize)
pc.global_setup(kernel_name='cubic_spline', structure='isotropic', num_neighbors=32)
pc.compute_smoothing()
pc.compute_density()  # required before interpolation

# --- build a regular 2D grid of query points (cell centers) ---
nx, ny = 64, 64
query_coords = create_grid_2d((nx, ny), boxsize)

# --- interpolate the scalar field to the grid points ---
grid_vals = pc.interpolate_fields(fields=values, query_positions=query_coords)
print(grid_vals.shape)
# reshape for plotting (create_grid_2d uses (nx, ny) ordering)
"""
grid_image = grid_vals.reshape((nx, ny)).T

# quick plot
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(grid_image, origin='lower', extent=(0, boxsize, 0, boxsize), cmap='viridis')
ax.scatter(positions[:, 0], positions[:, 1], s=2, color='k', alpha=0.4)
ax.set_title('Interpolated scalar field')
plt.colorbar(im, ax=ax)
plt.show()
"""

[smudgy] Initialized 2d PointCloud with 1000 particles in periodic box of size=[1. 1.]
[smudgy] Building kd-tree from particle positions
[smudgy] Computing smoothing lengths from 32 neighbors
[smudgy] Computing density using isotropic 'cubic_spline' kernel
[smudgy] Interpolating fields at query positions using isotropic 'cubic_spline' kernel
(1000, 1)


"\ngrid_image = grid_vals.reshape((nx, ny)).T\n\n# quick plot\nfig, ax = plt.subplots(figsize=(5, 4))\nim = ax.imshow(grid_image, origin='lower', extent=(0, boxsize, 0, boxsize), cmap='viridis')\nax.scatter(positions[:, 0], positions[:, 1], s=2, color='k', alpha=0.4)\nax.set_title('Interpolated scalar field')\nplt.colorbar(im, ax=ax)\nplt.show()\n"

Notes:
- `compute_density()` must be run after smoothing so interpolation has the
	correct normalization. The `global_setup` call configures kernel,
	structure and neighbor count used by the smoothing routine.

## Interpolating gradients of a field

To obtain gradients at the same query points, request gradient interpolation
from `interpolate_fields` (or use the convenience wrapper
`interpolate_gradient_fields`). The returned array has shape `(M, F, D)`
where `M` is the number of query points, `F` the number of fields and `D`
the spatial dimension.

In [6]:
# compute (vector) gradients on the same grid
grads = pc.interpolate_fields(values, grid_coords, compute_gradients=True)

# grads has shape (M, 1, 2) for a single scalar field in 2D
grad_vecs = grads[:, 0, :]  # shape (M, 2)
grad_mag = np.linalg.norm(grad_vecs, axis=1)
grad_mag_image = grad_mag.reshape((nx, ny)).T

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4))
ax0.imshow(grad_mag_image, origin='lower', extent=(0, boxsize, 0, boxsize), cmap='magma')
ax0.set_title('Gradient magnitude')
ax1.imshow(grid_image, origin='lower', extent=(0, boxsize, 0, boxsize), cmap='viridis')

# subsample arrows for clarity
step = max(1, nx // 16)
X = grid_coords.reshape((nx, ny, 2)).T[::step, ::step, 0]
Y = grid_coords.reshape((nx, ny, 2)).T[::step, ::step, 1]
U = grad_vecs[:, 0].reshape((nx, ny)).T[::step, ::step]
V = grad_vecs[:, 1].reshape((nx, ny)).T[::step, ::step]
ax1.quiver(X, Y, U, V, color='white', scale=50)
ax1.set_title('Interpolated field (with gradient vectors)')
plt.show()

NameError: name 'grid_coords' is not defined

This completes the minimal interpolation workflow. For multi‑field inputs
simply pass a `(N, F)` array for `fields` and the interpolation routines
will return `(M, F)` (values) or `(M, F, D)` (gradients).


### Next steps

- Use `deposit_to_grid` when you need conservative cell averages rather than
	pointwise interpolation.
- For anisotropic smoothing set `structure='anisotropic'` and compute
	smoothing tensors before interpolation.
